In [2]:
import regex as re
from collections import Counter, defaultdict

In [3]:
# remove all whitespace and punctuation, and convert to lowercase
def tokenize(text) -> list:
    text = text.strip() # remove leading and trailing whitespace
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    return text.split()

# Example usage
text = "Hello, World! This is a test."
print(f"Tokenized text : {tokenize(text)}")  

Tokenized text : ['hello', 'world', 'this', 'is', 'a', 'test']


In [4]:
# import the text of the book using nltk
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg   

# get the text of the book
text = gutenberg.raw('austen-emma.txt')
# tokenize the text
tokens = tokenize(text)
print(f"Number of tokens : {len(tokens)}")
print(f"First 10 tokens : {tokens[:10]}")

Number of tokens : 158132
First 10 tokens : ['emma', 'by', 'jane', 'austen', '1816', 'volume', 'i', 'chapter', 'i', 'emma']


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\govin\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [13]:
# model definition
class NGramModel:
    def __init__(self, n , smoothing=False):
        self.n = n
        self.ngrams = Counter() # count of n-grams
        self.contexts = defaultdict(Counter) # count of contexts and their following words 
        self.context_count = Counter() #  Context count for probability calculation
        self.vocab = set() # vocabulary of unique words
        self.smoothing = smoothing # whether to apply Laplace smoothing
    # Train the model on the given tokens
    # build n-grams and count their occurrences, as well as the contexts and their following words
    def train(self, tokens):
        for i in range(len(tokens) - self.n + 1):
            ngram = tuple(tokens[i:i+self.n])
            context = ngram[:-1]
            word = ngram[-1]
            self.ngrams[ngram] += 1
            self.contexts[context][word] += 1
            self.context_count[context] += 1
            self.vocab.add(word)

    # Get the probability of a word given a context
    def get_probability(self, context, word):
        if self.smoothing:
            # Apply Laplace smoothing
            return (self.contexts[context][word] + 1) / (self.context_count[context] + len(self.vocab))
        else:
            if self.context_count[context] == 0:
                return 0
            return self.contexts[context][word] / self.context_count[context]

    
    def predict(self, context)-> list:
        if context in self.contexts:
           return [{word: self.get_probability(context, word)} for word in self.contexts[context]]
        else:
            return None




In [18]:
# model instantiation and training
model = NGramModel(n=3, smoothing=False)
model.train(tokens)
print(f"Number of unique n-grams : {len(model.ngrams)}")
print(f"Number of unique contexts : {len(model.contexts)}")
print(f"Vocabulary size : {len(model.vocab)}")

Number of unique n-grams : 130458
Number of unique contexts : 68703
Vocabulary size : 9460


In [25]:
context = ('the', 'pride')
predicted_word = model.predict(context)
if predicted_word is not None:
    predicted_word.sort(key=lambda x: list(x.values())[0], reverse=True)
    predicted_word = predicted_word[:5]
    print(f"Predicted next words for context {context} : {predicted_word}")
else:
    print(f"No predictions available for context {context}")


Predicted next words for context ('the', 'pride') : [{'of': 0.8}, {'or': 0.2}]
